
# Robustness analyse dampingcoëfficiënten per DOF en constructie

Deze notebook doet per **DOF** en per **constructie** drie vergelijkingen voor elk experiment in de calibratie-Excel:

1. **Originele individuele fit** uit de Excel
2. **Gemiddelde coëfficiënten** van alle experimenten binnen dezelfde constructie + DOF
3. **Beste globale combinatie**: één van de best-fit combinaties uit de Excel die gemiddeld de laagste `nmrse_total` geeft over alle experimenten binnen dezelfde constructie + DOF

Outputs:
- overlayplot per experiment met 3 simulaties
- RMSE-vergelijkingsplot per constructie + DOF
- Excel met detailresultaten per experiment en samenvattingsbladen


In [1]:

# ============================================================
# 1. Imports
# ============================================================
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
from scipy.signal import find_peaks

import OrcFxAPI


In [2]:

# ============================================================
# 2. Instellingen
# ============================================================
# --- pas deze paden aan indien nodig ---
data_root = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\Decay_data\01_Rawdata")
orca_root = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA")

input_excel_path = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\Decay_calibration_outputs\calibrated_decay_coefficients_2304.xlsx")
output_dir = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\Decay_calibration_outputs\robustness_analysis")
output_dir.mkdir(parents=True, exist_ok=True)

plots_dir = output_dir / "plots"
overlay_dir = plots_dir / "overlay_per_experiment"
summary_plot_dir = plots_dir / "rmse_summary"
overlay_dir.mkdir(parents=True, exist_ok=True)
summary_plot_dir.mkdir(parents=True, exist_ok=True)

output_excel_path = output_dir / "robustness_analysis_results.xlsx"

vesselname = "floaters"
vesseltypename = "floatertype"  # pas aan indien nodig

# minima voor initiële amplitude
min_initial_amp = {
    "pitch": 3.0,
    "roll": 3.0,
    "heave": 0.8,
}

# peak detectie
prominence_map = {
    "pitch": 0.2,
    "roll": 0.2,
    "heave": 0.002,
}

# OrcaFlex modelbestanden
model_map = {
    "spring": {
        "40s": orca_root / "Harlequin_spring_40s.dat",
        "120s": orca_root / "Harlequin_spring_120s.dat",
    },
    "fixedwith": {
        "40s": orca_root / "Harlequin_fixed_40s.dat",
        "120s": orca_root / "Harlequin_fixed_120s.dat",
    },
    "fixedwithout": {
        "40s": orca_root / "Harlequin_fixed_40s.dat",
        "120s": orca_root / "Harlequin_fixed_120s.dat",
    }
}

# Alleen deze sheets worden verwerkt als ze bestaan én data bevatten.
sheets_to_process = ["pitch", "roll", "heave"]

# Plotinstellingen
FIGSIZE_OVERLAY = (14, 6)
FIGSIZE_SUMMARY = (12, 6)
DPI = 200


In [3]:

# ============================================================
# 3. Helper functies experimentdata
# ============================================================
def read_decay_signals(exp_path: Path, dof: str):
    filtered_map = {
        "pitch": "CroppedSignals/PITCH (LPF: 5.0 rad*s^-1)",
        "roll": "CroppedSignals/ROLL (LPF: 5.0 rad*s^-1)",
        "heave": "CroppedSignals/Z_COG (LPF: 5.0 rad*s^-1)",
    }

    unfiltered_map = {
        "pitch": "UnfilteredSignals/PITCH (unfiltered)",
        "roll": "UnfilteredSignals/ROLL (unfiltered)",
        "heave": "UnfilteredSignals/Z_COG (unfiltered)",
    }

    with h5py.File(exp_path, "r") as f:
        t_filt = np.array(f["CroppedSignals/time"])
        z_filt = np.array(f[filtered_map[dof]])

        t = np.array(f["UnfilteredSignals/time"])
        z = np.array(f[unfiltered_map[dof]])

    return t_filt, z_filt, t, z


def find_first_zero_crossing(t, z):
    z = np.asarray(z)

    for i in range(1, len(z)):
        if z[i - 1] == 0:
            return i - 1, t[i - 1]
        if z[i - 1] * z[i] < 0:
            return i, t[i]

    raise ValueError("Geen zero crossing gevonden.")


def find_initial_amplitude_from_zero_crossing(t, z, t_zero, search_window=20.0):
    mask = (t >= t_zero - search_window) & (t <= t_zero)

    if np.sum(mask) == 0:
        raise ValueError("Geen data gevonden in zoekvenster voor initiële amplitude.")

    t_win = t[mask]
    z_win = z[mask]

    i_local = np.argmax(np.abs(z_win))
    return t_win[i_local], z_win[i_local]


def classify_extremum_type(a_value):
    return "peak" if a_value >= 0 else "trough"


def find_first_two_same_type_extrema_after_reference(t, z, t_reference, extremum_type, prominence):
    t = np.asarray(t)
    z = np.asarray(z)

    if extremum_type == "peak":
        idx_all, _ = find_peaks(z, prominence=prominence)
    elif extremum_type == "trough":
        idx_all, _ = find_peaks(-z, prominence=prominence)
    else:
        raise ValueError(f"Onbekend extremum_type: {extremum_type}")

    if len(idx_all) < 2:
        raise ValueError(f"Minder dan twee {extremum_type}s gevonden in signaal.")

    idx_after_ref = idx_all[t[idx_all] >= t_reference]
    if len(idx_after_ref) < 2:
        raise ValueError(
            f"Niet genoeg {extremum_type}s gevonden na referentietijd {t_reference:.3f} s."
        )

    idx_A1 = int(idx_after_ref[0])
    idx_A2 = int(idx_after_ref[1])

    return {
        "idx_A1": idx_A1,
        "t_A1": float(t[idx_A1]),
        "a_A1": float(z[idx_A1]),
        "idx_A2": idx_A2,
        "t_A2": float(t[idx_A2]),
        "a_A2": float(z[idx_A2]),
        "type": extremum_type,
    }


def prepare_experiment(exp_path: Path, dof: str):
    t_filt, z_filt, t, z = read_decay_signals(exp_path, dof)

    idx_zero, t_zero = find_first_zero_crossing(t_filt, z_filt)

    t_release_guess, a_release_guess = find_initial_amplitude_from_zero_crossing(
        t, z, t_zero, search_window=20.0
    )

    if abs(a_release_guess) < min_initial_amp[dof]:
        raise ValueError(
            f"Initiële amplitude te klein voor {dof}: {a_release_guess:.4f} "
            f"(minimum {min_initial_amp[dof]:.4f})"
        )

    extremum_type = classify_extremum_type(a_release_guess)
    ext_info = find_first_two_same_type_extrema_after_reference(
        t=t,
        z=z,
        t_reference=t_release_guess,
        extremum_type=extremum_type,
        prominence=prominence_map[dof],
    )

    mask_decay = t >= ext_info["t_A2"]
    t_decay = t[mask_decay] - ext_info["t_A2"]
    z_decay = z[mask_decay]

    t_full_rel = t - ext_info["t_A2"]

    return {
        "exp_path": exp_path,
        "dof": dof,
        "t_filtered": t_filt,
        "z_filtered": z_filt,
        "t_full": t,
        "z_full": z,
        "t_full_rel": t_full_rel,
        "t_zero": t_zero,
        "t_A1": ext_info["t_A1"],
        "a_A1": ext_info["a_A1"],
        "t_A2": ext_info["t_A2"],
        "a_A2": ext_info["a_A2"],
        "extremum_type": extremum_type,
        "t_decay": t_decay,
        "z_decay": z_decay,
    }


In [4]:

# ============================================================
# 4. Helper functies score en extrema
# ============================================================
def get_peaks_and_troughs(t, z, prominence):
    peaks, _ = find_peaks(z, prominence=prominence)
    troughs, _ = find_peaks(-z, prominence=prominence)

    return {
        "peak_idx": peaks,
        "trough_idx": troughs,
        "t_peaks": t[peaks],
        "z_peaks": z[peaks],
        "t_troughs": t[troughs],
        "z_troughs": z[troughs],
    }


def match_extrema_by_order(z_ref, z_sim):
    n = min(len(z_ref), len(z_sim))
    if n == 0:
        return np.array([]), np.array([])
    return np.asarray(z_ref[:n]), np.asarray(z_sim[:n])


def compute_nrmse(z_ref, z_sim):
    z_ref = np.asarray(z_ref)
    z_sim = np.asarray(z_sim)

    if len(z_ref) == 0 or len(z_sim) == 0:
        return np.nan

    scale = np.maximum(np.abs(z_ref), 1e-6)
    return np.sqrt(np.mean(((z_sim - z_ref) / scale) ** 2))


def compute_score_from_extrema(exp_extrema, sim_extrema):
    z_ref_peaks, z_sim_peaks = match_extrema_by_order(exp_extrema["z_peaks"], sim_extrema["z_peaks"])
    z_ref_troughs, z_sim_troughs = match_extrema_by_order(exp_extrema["z_troughs"], sim_extrema["z_troughs"])

    nrmse_peaks = compute_nrmse(z_ref_peaks, z_sim_peaks)
    nrmse_troughs = compute_nrmse(z_ref_troughs, z_sim_troughs)

    vals = [v for v in [nrmse_peaks, nrmse_troughs] if np.isfinite(v)]
    score = np.mean(vals) if len(vals) > 0 else np.inf

    return score, nrmse_peaks, nrmse_troughs, len(z_ref_peaks), len(z_ref_troughs)


In [5]:

# ============================================================
# 5. Helper functies OrcaFlex
# ============================================================
def get_model_path(construction: str, dof: str) -> Path:
    sim_key = "120s" if dof in ["pitch", "roll"] else "40s"
    return model_map[construction][sim_key]


def set_initial_amplitude(vessel, dof: str, amplitude: float):
    vessel.InitialX = 0.0
    vessel.InitialY = 0.0
    vessel.InitialZ = 0.0
    vessel.InitialHeel = 0.0
    vessel.InitialTrim = 0.0
    vessel.InitialHeading = 0.0

    if dof == "pitch":
        vessel.InitialTrim = float(amplitude)
    elif dof == "roll":
        vessel.InitialHeel = float(amplitude)
    elif dof == "heave":
        vessel.InitialZ = float(amplitude)
    else:
        raise ValueError(f"Onbekende DOF: {dof}")


def set_damping(vesseltype, dof: str, lin_coeff: float, quad_coeff: float):
    dof = dof.lower()

    if dof == "pitch":
        vesseltype.OtherDampingLinearCoeffRy = lin_coeff
        vesseltype.OtherDampingQuadraticCoeffRy = quad_coeff
    elif dof == "roll":
        vesseltype.OtherDampingLinearCoeffRx = lin_coeff
        vesseltype.OtherDampingQuadraticCoeffRx = quad_coeff
    elif dof == "heave":
        vesseltype.OtherDampingLinearCoeffz = lin_coeff
        vesseltype.OtherDampingQuadraticCoeffz = quad_coeff
    else:
        raise ValueError(f"Onbekende DOF: {dof}")


def run_orcaflex_decay(model_path: Path, dof: str, initial_amplitude: float, lin_coeff: float, quad_coeff: float):
    model = OrcFxAPI.Model(str(model_path))
    vessel = model[vesselname]
    vesseltype = model[vesseltypename]

    set_initial_amplitude(vessel, dof, initial_amplitude)
    set_damping(vesseltype, dof, lin_coeff, quad_coeff)

    model.RunSimulation()

    if dof == "pitch":
        t = np.asarray(model.general.TimeHistory("Time"), dtype=float)
        z = np.asarray(vessel.TimeHistory("Rotation 2"), dtype=float)
    elif dof == "roll":
        t = np.asarray(model.general.TimeHistory("Time"), dtype=float)
        z = np.asarray(vessel.TimeHistory("Rotation 1"), dtype=float)
    elif dof == "heave":
        t = np.asarray(model.general.TimeHistory("Time"), dtype=float)
        z = np.asarray(vessel.TimeHistory("Z"), dtype=float)
    else:
        raise ValueError(f"Onbekende DOF: {dof}")

    return t, z


In [6]:

# ============================================================
# 6. Kernfuncties voor robustness analyse
# ============================================================
def sanitize_sheet_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    rename_map = {
        "nrmse_total": "nmrse_total",  # vang eventuele tikfout/variant op
        "initial amplitude": "intial amplitude",
        "initial_amplitude": "intial amplitude",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    required = [
        "construction", "dof", "exp_path", "file_name",
        "linear_damping", "quadratic_damping", "nmrse_total"
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Ontbrekende kolommen: {missing}")

    df = df.dropna(subset=["construction", "dof", "exp_path", "linear_damping", "quadratic_damping"])
    df["construction"] = df["construction"].astype(str).str.strip().str.lower()
    df["dof"] = df["dof"].astype(str).str.strip().str.lower()
    df["exp_path"] = df["exp_path"].astype(str).str.strip()
    df["file_name"] = df["file_name"].astype(str).str.strip()

    num_cols = [
        c for c in [
            "linear_damping", "quadratic_damping", "nmrse_total",
            "nrmse_peaks", "nrmse_troughs", "sim_init_amplitude",
            "sim_init_time", "intial amplitude"
        ] if c in df.columns
    ]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


def load_nonempty_sheets(excel_path: Path, requested_sheets=None):
    xls = pd.ExcelFile(excel_path)
    requested_sheets = requested_sheets or xls.sheet_names
    sheet_map = {}

    for sheet in requested_sheets:
        if sheet not in xls.sheet_names:
            continue
        df = pd.read_excel(excel_path, sheet_name=sheet)
        if df.dropna(how="all").empty:
            continue
        df = sanitize_sheet_df(df)
        if len(df) == 0:
            continue
        sheet_map[sheet] = df

    if not sheet_map:
        raise ValueError("Geen niet-lege sheets gevonden in de input Excel.")

    return sheet_map


def simulate_and_score(construction: str, dof: str, exp_info: dict, lin: float, quad: float):
    model_path = get_model_path(construction, dof)
    prominence = prominence_map[dof]

    t_sim_raw, z_sim_raw = run_orcaflex_decay(
        model_path=model_path,
        dof=dof,
        initial_amplitude=exp_info["a_A2"],
        lin_coeff=float(lin),
        quad_coeff=float(quad),
    )

    t_sim = np.asarray(t_sim_raw, dtype=float) - float(t_sim_raw[0])
    z_sim = np.asarray(z_sim_raw, dtype=float)

    exp_extrema = get_peaks_and_troughs(exp_info["t_decay"], exp_info["z_decay"], prominence=prominence)
    sim_extrema = get_peaks_and_troughs(t_sim, z_sim, prominence=prominence)

    score, nrmse_peaks, nrmse_troughs, n_peaks, n_troughs = compute_score_from_extrema(
        exp_extrema, sim_extrema
    )

    return {
        "lin": float(lin),
        "quad": float(quad),
        "nmrse_total": float(score),
        "nrmse_peaks": float(nrmse_peaks),
        "nrmse_troughs": float(nrmse_troughs),
        "n_peaks_used": int(n_peaks),
        "n_troughs_used": int(n_troughs),
        "t_sim": t_sim,
        "z_sim": z_sim,
        "sim_init_amplitude": float(z_sim[0]) if len(z_sim) else np.nan,
        "sim_init_time": float(t_sim[0]) if len(t_sim) else np.nan,
    }


def evaluate_candidate_set_on_group(group_df: pd.DataFrame, candidate_pairs: pd.DataFrame):
    construction = group_df["construction"].iloc[0]
    dof = group_df["dof"].iloc[0]

    prepared_experiments = {}
    for _, row in group_df.iterrows():
        exp_info = prepare_experiment(Path(row["exp_path"]), dof)
        prepared_experiments[row["exp_path"]] = exp_info

    candidate_rows = []
    for _, cand in candidate_pairs.iterrows():
        lin = float(cand["linear_damping"])
        quad = float(cand["quadratic_damping"])
        rmse_vals = []

        for exp_path, exp_info in prepared_experiments.items():
            sim_result = simulate_and_score(construction, dof, exp_info, lin, quad)
            rmse_vals.append(sim_result["nmrse_total"])

        candidate_rows.append({
            "construction": construction,
            "dof": dof,
            "linear_damping": lin,
            "quadratic_damping": quad,
            "mean_nmrse_total": float(np.mean(rmse_vals)),
            "std_nmrse_total": float(np.std(rmse_vals)),
            "n_experiments": len(rmse_vals),
        })

    candidate_eval_df = pd.DataFrame(candidate_rows).sort_values(
        ["mean_nmrse_total", "std_nmrse_total", "linear_damping", "quadratic_damping"]
    ).reset_index(drop=True)

    best_candidate = candidate_eval_df.iloc[0].to_dict()
    return prepared_experiments, candidate_eval_df, best_candidate


In [7]:

# ============================================================
# 7. Plotfuncties
# ============================================================
def plot_overlay_for_experiment(result_row: dict, exp_info: dict, save_path: Path):
    dof = result_row["dof"]
    prominence = prominence_map[dof]

    exp_ext = get_peaks_and_troughs(exp_info["t_decay"], exp_info["z_decay"], prominence=prominence)

    plt.figure(figsize=FIGSIZE_OVERLAY)
    plt.plot(exp_info["t_full_rel"], exp_info["z_full"], linewidth=2, label="Experiment")

    plt.plot(result_row["orig_t_sim"], result_row["orig_z_sim"], linewidth=2, label="Individuele fit")
    plt.plot(result_row["mean_t_sim"], result_row["mean_z_sim"], linewidth=2, label="Gemiddelde coeff.")
    plt.plot(result_row["best_t_sim"], result_row["best_z_sim"], linewidth=2, label="Beste globale combinatie")

    plt.scatter(exp_ext["t_peaks"], exp_ext["z_peaks"], s=40, marker="o", color = "#1f77b4")
    plt.scatter(exp_ext["t_troughs"], exp_ext["z_troughs"], s=40, marker="o", label="Extrema experiment")

    t_A1_rel = exp_info["t_A1"] - exp_info["t_A2"]
    dt = np.median(np.diff(exp_info["t_decay"])) if len(exp_info["t_decay"]) > 1 else 0.5
    x_start = t_A1_rel - 2.0 * dt
    x_end = max(
        result_row["orig_t_sim"][-1] if len(result_row["orig_t_sim"]) else 0,
        result_row["mean_t_sim"][-1] if len(result_row["mean_t_sim"]) else 0,
        result_row["best_t_sim"][-1] if len(result_row["best_t_sim"]) else 0,
    )
    plt.xlim(x_start - 10, x_end)

    if dof in ["pitch", "roll"]:
        plt.ylabel(f"{dof.capitalize()} [degrees]")
    else:
        plt.ylabel(f"{dof.capitalize()} [m]")
    plt.xlabel("Time [s]")
    plt.grid(True)
    plt.legend()
    plt.title(
        f"{result_row['construction']} | {dof} | {result_row['file_name']}"
        f"orig={result_row['orig_nmrse_total']:.5f}, mean={result_row['mean_nmrse_total']:.5f}, "
        f"best={result_row['best_nmrse_total']:.5f}"
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=DPI)
    plt.close()


def plot_rmse_summary(group_result_df: pd.DataFrame, save_path: Path):
    plot_df = group_result_df.copy().reset_index(drop=True)
    x = np.arange(len(plot_df))
    width = 0.25

    plt.figure(figsize=FIGSIZE_SUMMARY)
    plt.bar(x - width, plot_df["orig_nmrse_total"], width=width, label="Individuele fit")
    plt.bar(x, plot_df["mean_nmrse_total"], width=width, label="Gemiddelde coeff.")
    plt.bar(x + width, plot_df["best_nmrse_total"], width=width, label="Beste globale combinatie")

    plt.xticks(x, plot_df["file_name"], rotation=45, ha="right")
    plt.ylabel("NRMSE total")
    plt.xlabel("Experiment")
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend()
    plt.title(f"RMSE vergelijking | {plot_df['construction'].iloc[0]} | {plot_df['dof'].iloc[0]}")
    plt.tight_layout()
    plt.savefig(save_path, dpi=DPI)
    plt.close()


In [8]:

# ============================================================
# 8. Hoofdanalyse per constructie + DOF
# ============================================================
def analyse_group(group_df: pd.DataFrame):
    construction = group_df["construction"].iloc[0]
    dof = group_df["dof"].iloc[0]

    print(f"Analyseer construction={construction}, dof={dof}, n={len(group_df)}")

    candidate_pairs = (
        group_df[["linear_damping", "quadratic_damping"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    mean_lin = float(group_df["linear_damping"].mean())
    mean_quad = float(group_df["quadratic_damping"].mean())

    prepared_experiments, candidate_eval_df, best_candidate = evaluate_candidate_set_on_group(
        group_df=group_df,
        candidate_pairs=candidate_pairs,
    )

    result_rows = []

    for _, row in group_df.iterrows():
        exp_info = prepared_experiments[row["exp_path"]]

        orig_res = simulate_and_score(
            construction=construction,
            dof=dof,
            exp_info=exp_info,
            lin=float(row["linear_damping"]),
            quad=float(row["quadratic_damping"]),
        )

        mean_res = simulate_and_score(
            construction=construction,
            dof=dof,
            exp_info=exp_info,
            lin=mean_lin,
            quad=mean_quad,
        )

        best_res = simulate_and_score(
            construction=construction,
            dof=dof,
            exp_info=exp_info,
            lin=float(best_candidate["linear_damping"]),
            quad=float(best_candidate["quadratic_damping"]),
        )

        result_row = {
            "construction": construction,
            "dof": dof,
            "exp_path": row["exp_path"],
            "file_name": row["file_name"],
            "extremum_type": row["extremum_type"] if "extremum_type" in row else exp_info["extremum_type"],
            "a_A1": exp_info["a_A1"],
            "a_A2": exp_info["a_A2"],
            "t_A1": exp_info["t_A1"],
            "t_A2": exp_info["t_A2"],

            "input_linear_damping": float(row["linear_damping"]),
            "input_quadratic_damping": float(row["quadratic_damping"]),
            "input_nmrse_total": float(row["nmrse_total"]),
            "input_nrmse_peaks": float(row["nrmse_peaks"]) if "nrmse_peaks" in row and pd.notna(row["nrmse_peaks"]) else np.nan,
            "input_nrmse_troughs": float(row["nrmse_troughs"]) if "nrmse_troughs" in row and pd.notna(row["nrmse_troughs"]) else np.nan,

            "orig_lin": orig_res["lin"],
            "orig_quad": orig_res["quad"],
            "orig_nmrse_total": orig_res["nmrse_total"],
            "orig_nrmse_peaks": orig_res["nrmse_peaks"],
            "orig_nrmse_troughs": orig_res["nrmse_troughs"],

            "mean_lin": mean_lin,
            "mean_quad": mean_quad,
            "mean_nmrse_total": mean_res["nmrse_total"],
            "mean_nrmse_peaks": mean_res["nrmse_peaks"],
            "mean_nrmse_troughs": mean_res["nrmse_troughs"],
            "delta_mean_vs_orig": mean_res["nmrse_total"] - orig_res["nmrse_total"],

            "best_global_lin": float(best_candidate["linear_damping"]),
            "best_global_quad": float(best_candidate["quadratic_damping"]),
            "best_global_mean_nmrse": float(best_candidate["mean_nmrse_total"]),
            "best_nmrse_total": best_res["nmrse_total"],
            "best_nrmse_peaks": best_res["nrmse_peaks"],
            "best_nrmse_troughs": best_res["nrmse_troughs"],
            "delta_best_vs_orig": best_res["nmrse_total"] - orig_res["nmrse_total"],

            "orig_t_sim": orig_res["t_sim"],
            "orig_z_sim": orig_res["z_sim"],
            "mean_t_sim": mean_res["t_sim"],
            "mean_z_sim": mean_res["z_sim"],
            "best_t_sim": best_res["t_sim"],
            "best_z_sim": best_res["z_sim"],
        }
        result_rows.append(result_row)

        save_dir = overlay_dir / dof / construction
        save_dir.mkdir(parents=True, exist_ok=True)
        plot_overlay_for_experiment(
            result_row=result_row,
            exp_info=exp_info,
            save_path=save_dir / f"{Path(row['file_name']).stem}_overlay.png",
        )

    group_result_df = pd.DataFrame(result_rows)

    summary_save_dir = summary_plot_dir / dof
    summary_save_dir.mkdir(parents=True, exist_ok=True)
    plot_rmse_summary(
        group_result_df,
        summary_save_dir / f"rmse_summary_{construction}_{dof}.png",
    )

    summary_row = {
        "construction": construction,
        "dof": dof,
        "n_experiments": len(group_result_df),
        "mean_input_nmrse_total": float(group_result_df["input_nmrse_total"].mean()),
        "mean_recomputed_orig_nmrse_total": float(group_result_df["orig_nmrse_total"].mean()),
        "mean_meancoeff_nmrse_total": float(group_result_df["mean_nmrse_total"].mean()),
        "mean_bestglobal_nmrse_total": float(group_result_df["best_nmrse_total"].mean()),
        "mean_linear_damping": mean_lin,
        "mean_quadratic_damping": mean_quad,
        "best_global_lin": float(best_candidate["linear_damping"]),
        "best_global_quad": float(best_candidate["quadratic_damping"]),
        "best_global_candidate_mean_nmrse": float(best_candidate["mean_nmrse_total"]),
    }

    return group_result_df, candidate_eval_df, pd.DataFrame([summary_row])


In [9]:

# ============================================================
# 9. Draai volledige analyse over alle niet-lege sheets
# ============================================================
sheet_map = load_nonempty_sheets(input_excel_path, requested_sheets=sheets_to_process)
print(f"Gevonden sheets: {list(sheet_map.keys())}")

all_detail_results = {}
all_candidate_results = {}
summary_frames = []

for sheet_name, sheet_df in sheet_map.items():
    print("=" * 80)
    print(f"Start sheet: {sheet_name}")

    detail_frames = []
    candidate_frames = []

    for (construction, dof), group_df in sheet_df.groupby(["construction", "dof"], sort=True):
        group_df = group_df.reset_index(drop=True)
        group_result_df, candidate_eval_df, summary_df = analyse_group(group_df)

        detail_frames.append(group_result_df)
        candidate_frames.append(candidate_eval_df)
        summary_frames.append(summary_df)

    all_detail_results[sheet_name] = pd.concat(detail_frames, ignore_index=True) if detail_frames else pd.DataFrame()
    all_candidate_results[sheet_name] = pd.concat(candidate_frames, ignore_index=True) if candidate_frames else pd.DataFrame()

summary_df_all = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()

print("Analyse klaar.")
for sheet_name, df in all_detail_results.items():
    print(sheet_name, df.shape)


Gevonden sheets: ['pitch', 'roll', 'heave']
Start sheet: pitch
Analyseer construction=fixedwith, dof=pitch, n=3
Analyseer construction=fixedwithout, dof=pitch, n=4
Analyseer construction=spring, dof=pitch, n=3
Start sheet: roll
Analyseer construction=fixedwith, dof=roll, n=3
Analyseer construction=fixedwithout, dof=roll, n=4
Analyseer construction=spring, dof=roll, n=4
Start sheet: heave
Analyseer construction=fixedwith, dof=heave, n=3
Analyseer construction=fixedwithout, dof=heave, n=4
Analyseer construction=spring, dof=heave, n=4
Analyse klaar.
pitch (10, 38)
roll (11, 38)
heave (11, 38)


In [10]:

# ============================================================
# 10. Schrijf Excel-output
# ============================================================
with pd.ExcelWriter(output_excel_path, engine="openpyxl") as writer:
    for sheet_name, df in all_detail_results.items():
        safe_sheet_name = f"{sheet_name}_details"[:31]
        export_df = df.drop(columns=[
            "orig_t_sim", "orig_z_sim", "mean_t_sim", "mean_z_sim", "best_t_sim", "best_z_sim"
        ], errors="ignore")
        export_df.to_excel(writer, sheet_name=safe_sheet_name, index=False)

    for sheet_name, df in all_candidate_results.items():
        safe_sheet_name = f"{sheet_name}_candidates"[:31]
        df.to_excel(writer, sheet_name=safe_sheet_name, index=False)

    summary_df_all.to_excel(writer, sheet_name="summary", index=False)

print(f"Excel opgeslagen naar: {output_excel_path}")


Excel opgeslagen naar: C:\Users\verav\Desktop\Studie\Afstuderen\Decay_calibration_outputs\robustness_analysis\robustness_analysis_results.xlsx


In [11]:

# ============================================================
# 11. Snelle controle van de samenvatting
# ============================================================
summary_df_all


,construction,dof,n_experiments,mean_input_nmrse_total,mean_recomputed_orig_nmrse_total,mean_meancoeff_nmrse_total,mean_bestglobal_nmrse_total,mean_linear_damping,mean_quadratic_damping,best_global_lin,best_global_quad,best_global_candidate_mean_nmrse
0,fixedwith,pitch,3,0.073227,0.073227,0.146004,0.144593,277.777778,488.888889,303.333333,50.00,0.144593
1,fixedwithout,pitch,4,0.079643,0.079643,0.095504,0.093807,164.166667,1187.500000,166.666667,1200.00,0.093807
2,spring,pitch,3,0.041591,0.041591,0.050764,0.053716,320.000000,16.666667,323.333333,0.00,0.053716
3,fixedwith,roll,3,0.123009,0.123009,0.185677,0.191463,254.444444,583.333333,263.333333,350.00,0.191463
4,fixedwithout,roll,4,0.067383,0.067383,0.140327,0.147326,134.166667,249.166667,166.666667,0.00,0.147326
5,spring,roll,4,0.062646,0.062646,0.084445,0.088740,293.333333,120.000000,303.333333,50.00,0.088740
6,fixedwith,heave,3,0.209030,0.209030,0.262296,0.265406,1.833333,4.083333,2.000000,4.75,0.265406
7,fixedwithout,heave,4,0.141731,0.141731,0.161110,0.154296,0.687500,1.437500,0.000000,2.00,0.154296
8,spring,heave,4,0.141118,0.141118,0.244945,0.246130,3.000000,2.750000,2.000000,3.25,0.246130



# 12. Globale coëfficiënten per DOF

Deze stap neemt eerst per **constructie + DOF** het gemiddelde van de eerder gekalibreerde coëfficiënten en neemt daarna per **DOF** het gemiddelde over de drie constructies. Daarmee ontstaat één globale combinatie per DOF. Vervolgens worden alle experimenten opnieuw gesimuleerd met die globale DOF-coëfficiënten en wordt de RMSE percentueel vergeleken met de eerdere methodes.


In [12]:

# ============================================================
# 12. Globale coëfficiënten per DOF over alle constructies
# ============================================================
def pct_change(new_value, reference_value):
    if not np.isfinite(reference_value) or abs(reference_value) < 1e-12:
        return np.nan
    return 100.0 * (new_value - reference_value) / reference_value


def make_details_input_from_memory_or_excel():
    detail_frames = []

    if "all_detail_results" in globals() and isinstance(all_detail_results, dict) and len(all_detail_results) > 0:
        for sheet_name, df in all_detail_results.items():
            if df is None or df.empty:
                continue
            temp = df.copy()
            temp["source_sheet"] = sheet_name
            detail_frames.append(temp)
    else:
        xls = pd.ExcelFile(output_excel_path)
        for sheet_name in ["pitch_details", "roll_details", "heave_details"]:
            if sheet_name not in xls.sheet_names:
                continue
            temp = pd.read_excel(output_excel_path, sheet_name=sheet_name)
            if temp.dropna(how="all").empty:
                continue
            temp["source_sheet"] = sheet_name.replace("_details", "")
            detail_frames.append(temp)

    if not detail_frames:
        raise ValueError("Geen detailresultaten gevonden. Run eerst de robustness analyse of controleer output_excel_path.")

    details = pd.concat(detail_frames, ignore_index=True)
    details.columns = [str(c).strip() for c in details.columns]

    required = [
        "construction", "dof", "exp_path", "file_name",
        "input_linear_damping", "input_quadratic_damping",
        "orig_nmrse_total", "mean_nmrse_total", "best_nmrse_total"
    ]
    missing = [c for c in required if c not in details.columns]
    if missing:
        raise ValueError(f"Ontbrekende kolommen in detailresultaten: {missing}")

    details["construction"] = details["construction"].astype(str).str.strip().str.lower()
    details["dof"] = details["dof"].astype(str).str.strip().str.lower()
    details["exp_path"] = details["exp_path"].astype(str).str.strip()
    details["file_name"] = details["file_name"].astype(str).str.strip()

    num_cols = [
        "input_linear_damping", "input_quadratic_damping",
        "input_nmrse_total", "orig_nmrse_total", "mean_nmrse_total", "best_nmrse_total",
        "input_nrmse_peaks", "input_nrmse_troughs",
        "orig_nrmse_peaks", "orig_nrmse_troughs",
        "mean_nrmse_peaks", "mean_nrmse_troughs",
        "best_nrmse_peaks", "best_nrmse_troughs",
    ]
    for col in [c for c in num_cols if c in details.columns]:
        details[col] = pd.to_numeric(details[col], errors="coerce")

    return details


details_for_dof_global = make_details_input_from_memory_or_excel()

construction_mean_coeffs = (
    details_for_dof_global
    .groupby(["dof", "construction"], as_index=False)
    .agg(
        construction_mean_lin=("input_linear_damping", "mean"),
        construction_mean_quad=("input_quadratic_damping", "mean"),
        n_experiments=("file_name", "count"),
    )
)

dof_global_coeffs = (
    construction_mean_coeffs
    .groupby("dof", as_index=False)
    .agg(
        dof_global_lin=("construction_mean_lin", "mean"),
        dof_global_quad=("construction_mean_quad", "mean"),
        n_constructions=("construction", "nunique"),
        constructions=("construction", lambda x: ", ".join(sorted(x.astype(str).unique()))),
    )
)

display(dof_global_coeffs)


,dof,dof_global_lin,dof_global_quad,n_constructions,constructions
0,heave,1.840278,2.756944,3,"fixedwith, fixedwithout, spring"
1,pitch,253.981481,564.351852,3,"fixedwith, fixedwithout, spring"
2,roll,227.314815,317.500000,3,"fixedwith, fixedwithout, spring"


In [13]:

# ============================================================
# 13. Simuleer elk experiment opnieuw met globale DOF-coeffs
# ============================================================
dof_global_detail_rows = []

for _, coeff_row in dof_global_coeffs.iterrows():
    dof = coeff_row["dof"]
    global_lin = float(coeff_row["dof_global_lin"])
    global_quad = float(coeff_row["dof_global_quad"])

    dof_df = details_for_dof_global[details_for_dof_global["dof"] == dof].copy().reset_index(drop=True)
    prepared_experiments = {}

    print("=" * 80)
    print(f"Globale DOF-analyse: dof={dof}, lin={global_lin:.6g}, quad={global_quad:.6g}, n={len(dof_df)}")

    for _, row in dof_df.iterrows():
        exp_path = row["exp_path"]
        construction = row["construction"]

        if exp_path not in prepared_experiments:
            prepared_experiments[exp_path] = prepare_experiment(Path(exp_path), dof)

        exp_info = prepared_experiments[exp_path]

        global_res = simulate_and_score(
            construction=construction,
            dof=dof,
            exp_info=exp_info,
            lin=global_lin,
            quad=global_quad,
        )

        dof_global_detail_rows.append({
            "construction": construction,
            "dof": dof,
            "exp_path": exp_path,
            "file_name": row["file_name"],

            "dof_global_lin": global_lin,
            "dof_global_quad": global_quad,
            "dof_global_nmrse_total": global_res["nmrse_total"],
            "dof_global_nrmse_peaks": global_res["nrmse_peaks"],
            "dof_global_nrmse_troughs": global_res["nrmse_troughs"],

            "input_linear_damping": row["input_linear_damping"],
            "input_quadratic_damping": row["input_quadratic_damping"],

            "orig_nmrse_total": row["orig_nmrse_total"],
            "mean_constructie_nmrse_total": row["mean_nmrse_total"],
            "best_transfer_nmrse_total": row["best_nmrse_total"],

            "verval_pct_tov_individueel": pct_change(global_res["nmrse_total"], row["orig_nmrse_total"]),
            "verval_pct_tov_constructie_mean": pct_change(global_res["nmrse_total"], row["mean_nmrse_total"]),
            "verval_pct_tov_best_transfer": pct_change(global_res["nmrse_total"], row["best_nmrse_total"]),
        })

dof_global_details_df = pd.DataFrame(dof_global_detail_rows)
display(dof_global_details_df.head())


Globale DOF-analyse: dof=heave, lin=1.84028, quad=2.75694, n=11
Globale DOF-analyse: dof=pitch, lin=253.981, quad=564.352, n=10
Globale DOF-analyse: dof=roll, lin=227.315, quad=317.5, n=11


,construction,dof,exp_path,file_name,dof_global_lin,dof_global_quad,dof_global_nmrse_total,dof_global_nrmse_peaks,dof_global_nrmse_troughs,input_linear_damping,input_quadratic_damping,orig_nmrse_total,mean_constructie_nmrse_total,best_transfer_nmrse_total,verval_pct_tov_individueel,verval_pct_tov_constructie_mean,verval_pct_tov_best_transfer
0,fixedwith,heave,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,34224_03CB_02_003_008_01_Decay1.h5m,1.840278,2.756944,0.195089,0.094422,0.295756,0.0,4.25,0.173918,0.271433,0.332104,12.172706,-28.126485,-41.256803
1,fixedwith,heave,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,34224_03CB_02_003_009_01_Decay1.h5m,1.840278,2.756944,0.459929,0.539584,0.380274,2.0,4.75,0.252506,0.281106,0.252506,82.146011,63.613926,82.146011
2,fixedwith,heave,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,34224_03CB_02_003_010_01_Decay1.h5m,1.840278,2.756944,0.390300,0.458856,0.321745,3.5,3.25,0.200666,0.234347,0.211607,94.502982,66.548052,84.446219
3,fixedwithout,heave,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,34224_03CB_02_004_009_01_Decay1.h5m,1.840278,2.756944,0.266593,0.121795,0.411391,2.0,0.75,0.220579,0.236430,0.251626,20.860498,12.757465,5.948160
4,fixedwithout,heave,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,34224_03CB_02_004_010_01_Decay1.h5m,1.840278,2.756944,0.365655,0.281394,0.449915,0.0,2.00,0.119276,0.123905,0.119276,206.561979,195.109952,206.561979


In [14]:

# ============================================================
# 14. Samenvatting: gemiddelde RMSE en percentueel verval per DOF
# ============================================================
dof_global_summary_rows = []

for dof, sub in dof_global_details_df.groupby("dof", sort=True):
    global_lin = float(sub["dof_global_lin"].iloc[0])
    global_quad = float(sub["dof_global_quad"].iloc[0])

    mean_orig = float(sub["orig_nmrse_total"].mean())
    mean_constructie = float(sub["mean_constructie_nmrse_total"].mean())
    mean_best = float(sub["best_transfer_nmrse_total"].mean())
    mean_global = float(sub["dof_global_nmrse_total"].mean())

    dof_global_summary_rows.append({
        "dof": dof,
        "n_experiments": int(len(sub)),
        "dof_global_lin": global_lin,
        "dof_global_quad": global_quad,

        "mean_rmse_individueel": mean_orig,
        "mean_rmse_constructie_mean": mean_constructie,
        "mean_rmse_best_transfer": mean_best,
        "mean_rmse_dof_global": mean_global,

        "verval_pct_tov_individueel": pct_change(mean_global, mean_orig),
        "verval_pct_tov_constructie_mean": pct_change(mean_global, mean_constructie),
        "verval_pct_tov_best_transfer": pct_change(mean_global, mean_best),
    })

dof_global_summary_df = pd.DataFrame(dof_global_summary_rows).sort_values("dof").reset_index(drop=True)

beste_coeff_per_dof = dof_global_summary_df[
    [
        "dof",
        "dof_global_lin",
        "dof_global_quad",
        "mean_rmse_dof_global",
        "verval_pct_tov_individueel",
        "verval_pct_tov_constructie_mean",
        "verval_pct_tov_best_transfer",
    ]
].copy()

display(dof_global_summary_df)
display(beste_coeff_per_dof)


,dof,n_experiments,dof_global_lin,dof_global_quad,mean_rmse_individueel,mean_rmse_constructie_mean,mean_rmse_best_transfer,mean_rmse_dof_global,verval_pct_tov_individueel,verval_pct_tov_constructie_mean,verval_pct_tov_best_transfer
0,heave,11,1.840278,2.756944,0.159862,0.219191,0.217993,0.321622,101.186732,46.730892,47.537740
1,pitch,10,253.981481,564.351852,0.066303,0.097232,0.097016,0.184348,178.039910,89.595667,90.018575
2,roll,11,227.314815,317.500000,0.080832,0.132374,0.138059,0.258875,220.264876,95.562841,87.509934


,dof,dof_global_lin,dof_global_quad,mean_rmse_dof_global,verval_pct_tov_individueel,verval_pct_tov_constructie_mean,verval_pct_tov_best_transfer
0,heave,1.840278,2.756944,0.321622,101.186732,46.730892,47.537740
1,pitch,253.981481,564.351852,0.184348,178.039910,89.595667,90.018575
2,roll,227.314815,317.500000,0.258875,220.264876,95.562841,87.509934


In [15]:

# ============================================================
# 15. Voeg globale DOF-resultaten toe aan Excel-output
# ============================================================
with pd.ExcelWriter(output_excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    construction_mean_coeffs.to_excel(writer, sheet_name="construction_mean_coeffs", index=False)
    dof_global_coeffs.to_excel(writer, sheet_name="dof_global_coeffs", index=False)
    dof_global_details_df.to_excel(writer, sheet_name="dof_global_details", index=False)
    dof_global_summary_df.to_excel(writer, sheet_name="dof_global_summary", index=False)
    beste_coeff_per_dof.to_excel(writer, sheet_name="best_coeff_per_dof", index=False)

print(f"Globale DOF-resultaten toegevoegd aan: {output_excel_path}")


Globale DOF-resultaten toegevoegd aan: C:\Users\verav\Desktop\Studie\Afstuderen\Decay_calibration_outputs\robustness_analysis\robustness_analysis_results.xlsx
